In [1]:
# import modules
import datacube
import joblib
import numpy as np
import xarray as xr
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
from dea_tools.classification import sklearn_flatten, sklearn_unflatten 
from dea_tools.datahandling import load_ard
from dea_tools.plotting import display_map, rgb, xr_animation
from odc.algo import mask_cleanup
from datacube.utils.cog import write_cog
# Import required packages
import math
import folium
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.patheffects as PathEffects
import matplotlib.pyplot as plt
import xarray as xr
from matplotlib import colors as mcolours
from matplotlib.animation import FuncAnimation
from pathlib import Path
from pyproj import Transformer
from shapely.geometry import box
from skimage.exposure import rescale_intensity
from tqdm.auto import tqdm
from datetime import datetime, timedelta


import odc.geo.xr
from odc.ui import image_aspect
from dea_tools.spatial import add_geobox

In [2]:
import sklearn
print(sklearn.__version__)

1.5.2


In [3]:
dc = datacube.Datacube(app='fmc')

In [4]:
# import model
model = joblib.load('/home/jovyan/gdata1/projects/Hazards/Fuel_moisture/RF_AllBands_noLC_DEA_labeless.joblib')

In [5]:
def load_data(date, lon, lat):

    requested_date = datetime.strptime(date, '%d/%m/%Y')
    start_date = requested_date - timedelta(days=10)
    date_b = requested_date + timedelta(days=10)
    
    df = load_ard(dc=dc,
            products=['ga_s2am_ard_3', 'ga_s2bm_ard_3','ga_s2cm_ard_3'],
            measurements= ['nbart_blue','nbart_green','nbart_red','oa_fmask', 'nbart_red_edge_1' ,'nbart_red_edge_2' ,'nbart_red_edge_3','nbart_nir_1','nbart_nir_2','nbart_swir_2','nbart_swir_3', 'oa_nbart_contiguity'],
            mask_pixel_quality=False,
            x=lon,
            y=lat,
            resolution=(-20, 20),
            time = (str(start_date)[0:10], str(date_b)[0:10]),
            resampling={"*": "bilinear"},
            group_by='solar_day',
            output_crs= 'EPSG:3577')

    return df

def classify_FMC(data, model):
    """
    - data (xarray.Dataset): Sentinel-2 dataset containing required bands and optional multiple time steps. 
        
    - model (sklearn model): A pre-trained model for classification.
    
    Returns:
    - xarray.Dataset: A dataset containing the classified FMC results.
    
"""

    # Define masks. seperate cloud + shadow mask from water+ no_data mask becasue we want to do buffering of could+shadow but not water+no_data

    cloud_mask = (data.oa_fmask == 2) | (data.oa_fmask == 3)
    water_mask = (data.oa_fmask == 0) | (data.oa_nbart_contiguity == 0)

    #perfrom 1 pixel opening on cloud + shadow. three pixle dilation 
    better_cloud_mask = mask_cleanup(mask=cloud_mask, mask_filters=[("opening", 1),("dilation", 3)])

    #drop fmask from dataset before we classify
    data = data.drop(['oa_fmask', 'oa_nbart_contiguity'])

    data = data.where(data > -999, 0)

    #calculate NDVI and NDII

    data['ndii']=((data.nbart_nir_1-data.nbart_swir_2)/(data.nbart_nir_1+data.nbart_swir_2))
    data['ndvi']=((data.nbart_nir_1-data.nbart_red)/(data.nbart_nir_1+data.nbart_red))

    # change order of variables to be the same as the model expects
    data_neworder = data[['ndvi','ndii', 'nbart_blue','nbart_green','nbart_red','nbart_red_edge_1' ,'nbart_red_edge_2' ,'nbart_red_edge_3','nbart_nir_1','nbart_nir_2','nbart_swir_2','nbart_swir_3']] 
    
    #flattern the data using SKlearn_flatten
    data_flat = sklearn_flatten(data_neworder)
    
    #classify the data using the model
    print("predicting...")
    out_class = model.predict(data_flat)
    
    #return_classification to original shape
    #transpose because coords sideways when moving from Numpy to Xarray
    returned_result = sklearn_unflatten(out_class, data).transpose()
    
    #make results a dataset
    dataset_result = xr.Dataset({'LFMC':returned_result}, coords=data.coords, attrs=data.attrs)

    # print(data.attrs)
    
    #apply masks we generated before to classified data. it can be masked before we classify but then these pixels have a 0 value and it is better if it is 'no data'

    masked_data = dataset_result.where(~better_cloud_mask)
    masked_data = masked_data.where(~water_mask)
    
    #return
    return masked_data

In [6]:
#Open XML file 
locations_file = '/home/jovyan/gdata1/projects/Hazards/Fuel_moisture/FMC_estimates.csv'

locations = pd.read_csv(locations_file)

#filter out points with no coordinates
locations = locations[locations.x != 0]


locations_gdf = gpd.GeoDataFrame(
    locations , geometry=gpd.points_from_xy(locations.x, locations.y), crs="EPSG:4283"
)



In [7]:
# help(gpd.points_from_xy)

In [8]:
def accuracy_assessment(dataset, validation_points):
    """
    dataset: xarray data set to perform validaion on
    validation_points: the pandas dataframe with the points we want to validate
    
    """
    
    data_bucket={}
    
    for row in validation_points.iterfeatures():

        point_xval, point_yval = row['geometry']['coordinates']

        fmc_value = int(dataset.sel(x=point_xval, y=point_yval, method="nearest"))
            
        validation_points[row[date]] = fmc_value




In [9]:
#list all dates data has been collected

date_list = list(set(locations['Date'].tolist()))

#opend list of dats allready run
with open('processed_dates.txt', 'r') as file:
    
    # reading the file
    dates_done = file.read()
    
    # replacing end splitting the text 
    # when newline ('\n') is seen.
    complete_dates = dates_done.split("\n")
    print(complete_dates )
    file.close()

if len(complete_dates) > 1:
    date_list = date_list- complete_dates 

print(date_list)

['']
['2/02/2024', '10/10/2024', '26/12/2024', '14/10/2024', '29/12/2024', '1/01/2025', '19/10/2024', '29/10/2024', nan, '2/10/2024', '6/01/2025', '2/01/2025', '4/01/2025', '8/10/2024', '15/01/2025', '7/10/2024', '11/10/2024', '15/04/2025', '21/10/2024', '20/11/2024', '5/11/2024', '11/01/2025', '12/10/2024', '7/01/2025', '19/11/2024', '30/10/2024', '22/11/2024', '8/01/2025', '10/01/2025', '21/11/2024', '5/10/2024', '1/11/2024', '16/11/2024', '3/01/2025', '4/10/2024', '3/10/2024', '27/12/2024', '31/10/2024', '31/12/2024', '9/01/2025', '28/12/2024', '16/01/2025', '17/11/2024', '1/01/2024', '28/10/2024', '30/12/2024']


In [11]:
#make groups:
batch = 1

for date in date_list:
# date = '9/05/2025'
        
    day_subset = locations_gdf[locations_gdf['Date'] == date]
    # print(day_subset)

    #get list of all x cordinates in day subset
    x_list = list(set(day_subset['x'].tolist()))
    
    # Skip if there are no points for this date
    if day_subset.empty:
        pass
        # print('no points')
    else:
        # Get bounding box coordinates and add a buffer
        lon_min, lat_min, lon_max, lat_max = day_subset.total_bounds
        buffer = 0.04
        x = (lon_min - buffer, lon_max + buffer)
        y = (lat_min - buffer, lat_max + buffer)
        
        # try to load s2 data for this area for this day:
        try:
            sentinel_2_data = load_data(date=date, lon=x, lat=y)
    
            #RUN classification
            fmc = classify_FMC(sentinel_2_data, model)
    
        except Exception as e:
            # Handle the exception and print an error message
            print(f'Error loading data for {date}. Aborting. Error: {e}')
            # return
        #reproject validation points for data drill
        repo_day_subset = day_subset.to_crs('EPSG:3577')
    
        #create a data bucket (dictionary) to gather our data. this will be come a table
        data_bucket={}
    
       
        #itterate through features in shapfile
        for row in repo_day_subset.iterfeatures():
            #grab the shapfiles' attributes for our table of data
            mini_bucket = {}
            #mini bucked will be each row of the table
            mini_bucket['Site Name'] = row['properties']['Site Name']
            mini_bucket['Date'] = row['properties']['Date']
            mini_bucket['x'] = row['properties']['x']
            mini_bucket['y'] = row['properties']['y']
    
            point_xval, point_yval = row['geometry']['coordinates']
    
            for layers in sentinel_2_data.time:
                #now conduct pixel drill for our points for each time scene loaded
                dataset = fmc.LFMC.sel(time = layers)
        
                date_str = str(layers.data)[0:10]
                fmc_value = (dataset.sel(x=point_xval, y=point_yval, method="nearest")).data
        
                mini_bucket[date_str] = fmc_value
                #add to our row
            
            data_bucket[row['id']] = mini_bucket
            #add row to table
    
        #trun bucket into table
        output_table = pd.DataFrame.from_dict(data_bucket, orient='index')
        #save
        output_table.to_csv(f'/home/jovyan/gdata1/projects/Hazards/Fuel_moisture/FMC_estimates_200525_{batch}_subset.csv')
        batch = batch + 1

        #add date to a txt file so we know what we have done allready

        with open('processed_dates.txt', 'a') as file:
            file.write(f'\n {date}')


Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps


/env/lib/python3.10/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(


predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 8 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 8 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 5 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 8 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard

Error opening source dataset: s3://dea-public-data/baseline/ga_s2am_ard_3/55/HGA/2024/10/15/20241015T005850/ga_s2am_nbart_3-2-1_55HGA_2024-10-15_final_band06.tif


Error loading data for 7/10/2024. Aborting. Error: Read failed. See previous exception for details.
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 9 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 8 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 5 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 9 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 9 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 8 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps


...

predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 8 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 9 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 5 time steps


Error opening source dataset: s3://dea-public-data/baseline/ga_s2bm_ard_3/55/HDV/2024/10/03/20241003T025213/ga_s2bm_nbart_3-2-1_55HDV_2024-10-03_final_band04.tif


predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 8 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 9 time steps


Error opening source dataset: s3://dea-public-data/baseline/ga_s2bm_ard_3/55/HEV/2024/10/03/20241003T025213/ga_s2bm_nbart_3-2-1_55HEV_2024-10-03_final_band08.tif
Error opening source dataset: s3://dea-public-data/baseline/ga_s2am_ard_3/55/HCU/2024/10/11/20241011T014342/ga_s2am_nbart_3-2-1_55HCU_2024-10-11_final_band08.tif


predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 7 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 5 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard

In [24]:
output_table

,Site Name,Date,x,y,2025-04-28,2025-05-03,2025-05-08,2025-05-13
39,t5.1,6/05/2025,150.580527,-34.893890,109.84765877932985,198.14917706583927,nan,nan
41,t5.2,6/05/2025,150.580275,-34.893871,123.21027257187961,108.48024571582825,nan,nan
44,t5.3,6/05/2025,150.579953,-34.893835,107.40005298393854,106.19932986220643,nan,nan
45,t5.4,6/05/2025,150.579658,-34.893791,104.02372218249566,106.5289883073546,nan,nan
48,t5.5,6/05/2025,150.579293,-34.893772,105.80795906898898,105.81496427719298,nan,nan
49,t6.5,6/05/2025,150.579306,-34.893556,105.16293838267367,105.46206730153143,nan,nan
52,t6.4,6/05/2025,150.579534,-34.893470,103.48000393620968,105.273923337782,nan,nan
54,t6.2,6/05/2025,150.580116,-34.893409,105.76654429627514,108.14113867794116,nan,nan
56,t6.1,6/05/2025,150.580381,-34.893378,104.9667540626592,104.85461013722814,nan,nan
59,t7.1,6/05/2025,150.580477,-34.893237,117.52985808886842,112.05648532486533,nan,nan


In [25]:
sentinel_2_data.time.data[0]

numpy.datetime64('2025-04-28T00:06:27.592138000')

In [28]:

import pandas as pd
import os

def combine_csv_files(folder_path):
# List to hold dataframes
    df_list = []
    
    # Iterate over all files in the folder
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.csv'):
            file_path = os.path.join(folder_path, file_name)
            df = pd.read_csv(file_path)
            df_list.append(df)
    
    # Combine all dataframes
    combined_df = pd.concat(df_list, ignore_index=True)
    return combined_df

# use
folder_path = '/home/jovyan/gdata1/projects/Hazards/Fuel_moisture/generated_validation_data'# Replace with your actual folder path
combined_df = combine_csv_files(folder_path)
print(combined_df)

     Unnamed: 0 Site Name        Date           x          y  2025-04-28  \
0           120      11.4   8/05/2025  150.579470 -34.896529  103.924509   
1           123      11.4   8/05/2025  150.579696 -34.896485  105.459094   
2           126      11.2   8/05/2025  150.580075 -34.896339  105.438842   
3           128      11.1   8/05/2025  150.580292 -34.896297  105.462207   
4           130      12.1   8/05/2025  150.580372 -34.896592  107.774262   
..          ...       ...         ...         ...        ...         ...   
212         208      16.5  28/03/2025  150.583508 -34.897885         NaN   
213         210      16.4  28/03/2025  150.583513 -34.898054         NaN   
214         212      16.3  28/03/2025  150.583420 -34.898234         NaN   
215         214      16.2  28/03/2025  150.583490 -34.898494         NaN   
216         217      16.1  28/03/2025  150.583438 -34.898691         NaN   

     2025-05-03  2025-05-08  2025-05-13  2024-03-19  2024-03-24  2024-03-29  \
0    104

In [30]:
combined_df.to_csv(f'/home/jovyan/gdata1/projects/Hazards/Fuel_moisture/SSFS_fmc_ws2C_all_points.csv')